# Part 8: 프레임워크 비교 분석**소요 시간**: 약 1.5시간**난이도**: ⭐⭐⭐ (중급)**목표**: MS GraphRAG, LightRAG, fast-graphrag, LlamaIndex를 비교하고, 프로젝트에 맞는 프레임워크를 선택할 수 있다.---## 학습 순서1. 환경 설정2. MS GraphRAG 아키텍처3. LightRAG / fast-graphrag4. LlamaIndex PropertyGraph v25. 프레임워크별 벤치마크6. 선택 가이드 + 하이브리드 전략7. 연습 문제

---## 1. 환경 설정### 패키지 설치 및 Neo4j 연결

In [ ]:
import os, json, timefrom dotenv import load_dotenvfrom neo4j import GraphDatabaseload_dotenv()load_dotenv(dotenv_path="../.env")NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))try:    driver.verify_connectivity()    print("[OK] Neo4j 연결 성공!")except Exception as e:    print(f"[ERROR] 연결 실패: {e}")def run_query(query, parameters=None, print_result=True):    with driver.session() as session:        result = session.run(query, parameters or {})        records = [record.data() for record in result]        if print_result:            for i, rec in enumerate(records, 1):                print(f"  [{i}] {rec}")            if not records:                print("  (결과 없음)")        return recordsprint("[OK] 환경 설정 완료")

---## 2. MS GraphRAG 아키텍처Microsoft GraphRAG는 **Leiden 커뮤니티 탐지 + 계층적 요약**을 결합한 프레임워크입니다.### 핵심 구조```문서 → 엔티티 추출 → 그래프 구축 → Leiden 커뮤니티                                         ↓                              커뮤니티별 요약 생성                                         ↓                              Global / Local / DRIFT Search```### 3가지 검색 모드| 모드 | 설명 | 적합한 질문 ||------|------|------------|| **Local Search** | 관련 엔티티 주변 탐색 | "삼성전자 CEO는?" || **Global Search** | 전체 커뮤니티 요약 활용 | "반도체 산업 트렌드는?" || **DRIFT Search** | Local + 커뮤니티 컨텍스트 | "삼성의 AI 전략은?" |> **핵심**: Global Search가 MS GraphRAG의 차별점. 다른 프레임워크에는 없는 기능.

In [ ]:
# MS GraphRAG 설치 및 사용 예시 (의사 코드)ms_graphrag_example = '''# 설치pip install graphrag# 초기화graphrag init --root ./my-project# 인덱싱 (시간 + 비용 소모)graphrag index --root ./my-project# 검색graphrag query --root ./my-project --method local "삼성전자 CEO는?"graphrag query --root ./my-project --method global "반도체 산업 트렌드는?"'''print("=== MS GraphRAG 사용 예시 ===")print(ms_graphrag_example)print("[주의] 인덱싱에 시간과 비용이 많이 듭니다.")print("  50개 문서 기준: ~18분, ~$12 (GPT-4 사용 시)")

---## 3. LightRAG / fast-graphrag### 3.1 LightRAG — 1/100 비용LightRAG는 MS GraphRAG의 **경량화 버전**입니다.| 비교 | MS GraphRAG | LightRAG ||------|-------------|----------|| 인덱싱 비용 | $11.77 | **$0.12** || 인덱싱 시간 | 18분 | **2분** || 쿼리 속도 | 3.2초 | **0.8초** || 정확도 | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ || 커뮤니티 요약 | O | X |### 3.2 fast-graphrag — PageRank 기반fast-graphrag는 **Personalized PageRank**를 활용해 검색 속도를 극대화합니다.- MS GraphRAG 대비 **27배 빠른** 쿼리 응답- 커뮤니티 탐지 없이 PageRank로 중요도 계산- 비용: MS GraphRAG의 1/5 수준

In [ ]:
# 프레임워크별 코드 비교lightrag_example = '''from lightrag import LightRAGengine = LightRAG(model="gpt-3.5-turbo")engine.index_directory("./data")result = engine.query("삼성전자 투자 기관은?")'''fast_graphrag_example = '''from fast_graphrag import FastGraphRAGengine = FastGraphRAG(model="gpt-4-turbo")engine.index("./data")result = engine.query("삼성전자 투자 기관은?")'''print("=== LightRAG 예시 ===")print(lightrag_example)print("\n=== fast-graphrag 예시 ===")print(fast_graphrag_example)

---## 4. LlamaIndex PropertyGraph v2LlamaIndex 0.11+에서 지원하는 PropertyGraph는 **Neo4j 통합**이 잘 되어 있습니다.**장점:**- 10줄 코드로 프로토타입 가능- Neo4j + 벡터 스토어 통합- LlamaIndex 에코시스템 활용**단점:**- 온톨로지 커스터마이징 제한- 디버깅이 어려움 (블랙박스)> **실무 팁**: POC는 LlamaIndex로 빠르게 만들고, 프로덕션은 직접 구현으로 마이그레이션

In [ ]:
# LlamaIndex PropertyGraph 사용 예시 (의사 코드)llamaindex_example = '''from llama_index.core import PropertyGraphIndex, SimpleDirectoryReaderfrom llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore# Neo4j 연결graph_store = Neo4jPropertyGraphStore(    url="bolt://localhost:7687",    username="neo4j",    password="password123")# 문서 로드 + 인덱스 생성 (단 3줄!)documents = SimpleDirectoryReader("./data").load_data()index = PropertyGraphIndex.from_documents(    documents,    property_graph_store=graph_store,    show_progress=True)# 검색query_engine = index.as_query_engine()response = query_engine.query("삼성전자에 투자한 기관은?")print(response)'''print("=== LlamaIndex PropertyGraph v2 ===")print(llamaindex_example)print("\n[주의] llama-index-graph-stores-neo4j 패키지 필요")

---## 5. 프레임워크별 벤치마크### 벤치마크 결과 (50개 문서 기준)| 프레임워크 | 인덱싱 시간 | 인덱싱 비용 | 쿼리 속도 | 정확도 ||-----------|------------|------------|----------|--------|| **MS GraphRAG** | 18m 32s | $11.77 | 3.2초 | ⭐⭐⭐⭐⭐ || **LightRAG** | 2m 15s | $1.20 | 0.8초 | ⭐⭐⭐ || **fast-graphrag** | 3m 40s | $2.05 | 0.4초 | ⭐⭐⭐⭐ || **LlamaIndex** | 5m 10s | $3.50 | 1.2초 | ⭐⭐⭐⭐ || **직접 구현** | 수동 | $0 | 0.1초 | 도메인 의존 |> **핵심**: 프레임워크 선택은 기술적 문제가 아니라 비즈니스 우선순위 문제

In [ ]:
# 벤치마크 결과 시각화benchmark = {    "MS GraphRAG": {"cost": 11.77, "speed": 3.2, "accuracy": 5},    "LightRAG":    {"cost": 1.20,  "speed": 0.8, "accuracy": 3},    "fast-graphrag":{"cost": 2.05, "speed": 0.4, "accuracy": 4},    "LlamaIndex":  {"cost": 3.50,  "speed": 1.2, "accuracy": 4},}print("=== 벤치마크 결과 (50개 문서 기준) ===")print(f"{'프레임워크':<18} {'비용($)':<10} {'쿼리속도(초)':<14} {'정확도'}")print("-" * 60)for name, data in benchmark.items():    stars = "★" * data["accuracy"] + "☆" * (5 - data["accuracy"])    print(f"{name:<18} ${data['cost']:<8.2f} {data['speed']:<12.1f}  {stars}")

---## 6. 선택 가이드 + 의사결정 트리### 의사결정 트리```❓ Multi-hop 질문 필요?  ├─ NO → 벡터 RAG로 충분  └─ YES ↓      ❓ 온톨로지 맞춤 설계 필요?        ├─ YES → 직접 구현 (Part 1-7)        └─ NO ↓            ❓ 비용 제약 있음?              ├─ YES → LightRAG              └─ NO ↓                  ❓ 정확도 vs 속도?                    ├─ 정확도 → MS GraphRAG                    └─ 속도 → fast-graphrag```### 하이브리드 전략 (실무 권장)| Phase | 기간 | 프레임워크 | 목표 ||-------|------|----------|------|| Phase 1 | 2주 | LlamaIndex | 빠른 POC || Phase 2 | 2개월 | 직접 구현 | 프로덕션 마이그레이션 || Phase 3 | 지속 | 최적화 | 인덱스 튜닝, 비용 절감 |

In [ ]:
# 대화형 프레임워크 추천기def recommend_framework():    print("=== GraphRAG 프레임워크 선택 가이드 ===\n")    q1 = input("1. Multi-hop 질문이 필요한가요? (y/n) ").strip().lower()    if q1 != 'y':        return "Vector RAG (FAISS + LangChain)로 충분합니다."    q2 = input("2. 온톨로지를 직접 설계해야 하나요? (y/n) ").strip().lower()    if q2 == 'y':        return "직접 구현 (Part 1-7 방식) - Neo4j + Cypher 완전 제어"    q3 = input("3. 비용 제약이 강한가요? (y/n) ").strip().lower()    if q3 == 'y':        return "LightRAG - 1/100 비용"    q4 = input("4. 정확도 vs 속도? (accuracy/speed) ").strip().lower()    if q4 == 'accuracy':        return "MS GraphRAG - Global Search 최고 정확도"    return "fast-graphrag - 27배 빠른 Personalized PageRank"# 주석 해제 후 실행# print(recommend_framework())print("[대화형 추천기] 위 함수의 주석을 해제하고 실행하세요.")

---## 7. 연습 문제### 문제 1: 프레임워크 선택 시나리오다음 3가지 시나리오에서 어떤 프레임워크를 선택할지 이유와 함께 작성하세요.**시나리오 A**: 스타트업 POC (예산 $500, 기간 2주, 100개 문서)**시나리오 B**: 대기업 프로덕션 (예산 $10K/월, 100만 문서, 맞춤 온톨로지)**시나리오 C**: 실시간 챗봇 (예산 $2K/월, 1만 문서, 200ms 응답)

In [ ]:
# [연습 1] 시나리오별 프레임워크 선택scenario_a = "시나리오 A 선택: ???  이유: "scenario_b = "시나리오 B 선택: ???  이유: "scenario_c = "시나리오 C 선택: ???  이유: "print(scenario_a)print(scenario_b)print(scenario_c)

### 문제 2: 하이브리드 전략 설계"POC는 LlamaIndex, 프로덕션은 직접 구현"으로 마이그레이션하는 전략을 설계하세요.

In [ ]:
# [연습 2] 하이브리드 전략migration_plan = {    "phase1_poc": {"framework": "???", "duration": "2주", "features": []},    "phase2_migration": {"custom_impl": [], "keep_from_poc": [], "risks": []},    "phase3_production": {"monitoring": [], "optimization": []},}# TODO: 위 템플릿을 채우세요print("하이브리드 전략을 설계해보세요!")

---## 8. 정리| 프레임워크 | 핵심 | 선택 기준 ||-----------|------|----------|| MS GraphRAG | Leiden + Global Search | 정확도 > 비용 || LightRAG | 1/100 비용 | 비용 제약 강함 || fast-graphrag | PageRank 27배 빠름 | 속도 우선 || LlamaIndex | 10줄 프로토타입 | 빠른 POC || 직접 구현 | 완전 제어 | 프로덕션, 맞춤 온톨로지 |### 다음 Part 9 미리보기: 그래프 알고리즘 + RAG 고도화Part 9에서는 Neo4j GDS로 PageRank, 커뮤니티 탐지를 실행하고 RAG 파이프라인에 통합합니다.

In [ ]:
# 세션 정리# driver.close()# print("[OK] Neo4j 드라이버 종료 완료")print("실습을 마칩니다. 수고하셨습니다!")